In [88]:
import numpy as np, os
import librosa, librosa.display, music21
from music21 import note, stream
import soundfile as sf
import pandas as pd

In [89]:
import joblib
model = joblib.load("rf_13_classifier.pkl")

In [90]:
'''
files = [
    "keyboard_acoustic_000-050-127.wav",
    "keyboard_acoustic_003-044-127.wav",
    "keyboard_acoustic_007-048-075.wav",
]
'''
def create_audio_file(files):
    combined_audio=[]
    for f in files:
        path = os.path.join("nsynth-train/audio",f)
        y,sr = librosa.load(path, sr=16000)

        '''
        adding silences to make note changes more distinct
        '''
        silence=np.zeros(int(0.2*sr)) #200ms gap

        combined_audio.append(y)
        combined_audio.append(silence)
    return np.concatenate(combined_audio)

'''
y_combined = create_audio_file(files)
'''




'\ny_combined = create_audio_file(files)\n'

In [91]:
'''import soundfile as sf
sf.write("./seq/seq3.wav",y_combined,sr=16000)'''

'import soundfile as sf\nsf.write("./seq/seq3.wav",y_combined,sr=16000)'

In [92]:
def predict_frames(y_audio, sr):
    frame_length=int(0.05*sr) #50ms frames
    hop_length=frame_length

    pitches=[]
    for i in range(0,len(y_audio), hop_length):
        frame=y_audio[i:i+frame_length]
        if len(frame)<frame_length:
            continue

        feature = np.hstack((
            np.mean(librosa.feature.mfcc(y=frame, sr=sr, n_mfcc=13).T, axis=0),
            np.mean(librosa.feature.chroma_stft(y=frame, sr=sr).T, axis=0),
            np.mean(librosa.feature.spectral_centroid(y=frame, sr=sr)),
            np.mean(librosa.feature.zero_crossing_rate(frame))
        )).reshape(1,-1)

        pitch=model.predict(feature)[0]
        pitches.append(pitch)

    return pitches

In [93]:
def segment_audio_onsets(y, sr):
    global files
    # detect note boundaries
    onset_frames = librosa.onset.onset_detect(
    y=y,
    sr=sr,
    backtrack=True,
    delta=0.05,
    wait=3
)

    onset_samples = librosa.frames_to_samples(onset_frames)

    segments = []

    for i in range(len(onset_samples)-1):
        start = onset_samples[i]
        end = onset_samples[i+1]
        segments.append((start, end))

    # last segment
    segments.append((onset_samples[-1], len(y)))
    print("Detected segments:", len(segments),"notes",len(files))
    if len(segments) != len(files):
        print("WARNING: possible segmentation issue")
    return segments

In [94]:
def predict_segments(y, sr, segments):

    notes = []
    durations = []

    for start, end in segments:

        segment = y[start:end]

        # skip silence
        if np.mean(np.abs(segment)) < 0.005:
            continue

        if len(segment) < 500:
            continue

        feature = np.hstack((
            np.mean(librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13).T, axis=0),
            np.mean(librosa.feature.chroma_stft(y=segment, sr=sr).T, axis=0),
            np.mean(librosa.feature.spectral_centroid(y=segment, sr=sr)),
            np.mean(librosa.feature.zero_crossing_rate(segment))
        )).reshape(1, -1)

        pitch = model.predict(feature)[0]

        duration = (end - start) / sr

        notes.append(pitch)
        durations.append(duration)

    return notes, durations

In [95]:
def merge_consecutive(notes, durations):

    merged_notes = []
    merged_durations = []

    for n, d in zip(notes, durations):

        if merged_notes:
            prev_n = merged_notes[-1]
            prev_d = merged_durations[-1]

            # merge ONLY if:
            # same note AND clearly smaller fragment
            if n == prev_n and d < 0.6 * prev_d:
                merged_durations[-1] += d
                continue

        merged_notes.append(n)
        merged_durations.append(d)

    return merged_notes, merged_durations

In [96]:
def seconds_to_beats(d, bpm=120):
    return d / (60 / bpm)

In [97]:
def quantize_duration(d):
    vals=[0.25,0.5,1,2,4,8]
    return min(vals, key=lambda x: abs(x-d))

def create_seq_sheet(notes, durations,filename):

    s = stream.Stream()

    for p, d in zip(notes, durations):

        n = note.Note()
        n.pitch.midi = int(p)
        beats=seconds_to_beats(d,bpm=120)
        n.quarterLength=quantize_duration(beats)

        s.append(n)

    s.write("musicxml", "seq/"+filename+".xml")

    print("Sheet music created at seq/"+filename+".xml")

In [98]:
'''y, sr = librosa.load("seq/seq3.wav", sr=16000)

segments = segment_audio_onsets(y, sr)

notes, durations = predict_segments(y, sr, segments)

create_seq_sheet(notes, durations,"sheet3")
print()
print(sum(durations), len(y)/sr)'''

'y, sr = librosa.load("seq/seq3.wav", sr=16000)\n\nsegments = segment_audio_onsets(y, sr)\n\nnotes, durations = predict_segments(y, sr, segments)\n\ncreate_seq_sheet(notes, durations,"sheet3")\nprint()\nprint(sum(durations), len(y)/sr)'

In [99]:
'''df=pd.DataFrame({"notes": notes, "durations": durations})
df.to_csv("seq/notes_durations3.csv", index=True)'''

'df=pd.DataFrame({"notes": notes, "durations": durations})\ndf.to_csv("seq/notes_durations3.csv", index=True)'

# TESTS

In [100]:
''' 
test 1 -> 3 notes with 2 repeated
'''
'''files = [
    "keyboard_acoustic_000-044-127.wav",
    "keyboard_acoustic_000-044-127.wav",
    "keyboard_acoustic_003-048-127.wav",
]

y_combined = create_audio_file(files)
sf.write("./seq/test1.wav",y_combined,16000)
y, sr = librosa.load("seq/test1.wav", sr=16000)
segments = segment_audio_onsets(y, sr)
notes, durations = predict_segments(y, sr, segments)
notes, durations = merge_consecutive(notes, durations)
create_seq_sheet(notes, durations,"test1_sheet")
print(sum(durations), len(y)/sr)
df=pd.DataFrame({"notes": notes, "durations": durations})
df.to_csv("seq/test1_notes.csv", index=True)
'''

'files = [\n    "keyboard_acoustic_000-044-127.wav",\n    "keyboard_acoustic_000-044-127.wav",\n    "keyboard_acoustic_003-048-127.wav",\n]\n\ny_combined = create_audio_file(files)\nsf.write("./seq/test1.wav",y_combined,16000)\ny, sr = librosa.load("seq/test1.wav", sr=16000)\nsegments = segment_audio_onsets(y, sr)\nnotes, durations = predict_segments(y, sr, segments)\nnotes, durations = merge_consecutive(notes, durations)\ncreate_seq_sheet(notes, durations,"test1_sheet")\nprint(sum(durations), len(y)/sr)\ndf=pd.DataFrame({"notes": notes, "durations": durations})\ndf.to_csv("seq/test1_notes.csv", index=True)\n'

In [101]:
''' 
test 2 -> 5 notes
'''

'''files = [
    "keyboard_acoustic_000-044-127.wav",
    "keyboard_acoustic_003-048-127.wav",
    "keyboard_acoustic_007-050-075.wav",
    "keyboard_acoustic_011-053-127.wav",
    "keyboard_acoustic_018-045-127.wav"
]

y_combined = create_audio_file(files)
sf.write("./seq/test2.wav",y_combined,16000)
y, sr = librosa.load("seq/test2.wav", sr=16000)
segments = segment_audio_onsets(y, sr)
notes, durations = predict_segments(y, sr, segments)
notes, durations = merge_consecutive(notes, durations)
create_seq_sheet(notes, durations,"test2_sheet")
print(sum(durations), len(y)/sr)
df=pd.DataFrame({"notes": notes, "durations": durations})
df.to_csv("seq/test2_notes.csv", index=True)
'''

'files = [\n    "keyboard_acoustic_000-044-127.wav",\n    "keyboard_acoustic_003-048-127.wav",\n    "keyboard_acoustic_007-050-075.wav",\n    "keyboard_acoustic_011-053-127.wav",\n    "keyboard_acoustic_018-045-127.wav"\n]\n\ny_combined = create_audio_file(files)\nsf.write("./seq/test2.wav",y_combined,16000)\ny, sr = librosa.load("seq/test2.wav", sr=16000)\nsegments = segment_audio_onsets(y, sr)\nnotes, durations = predict_segments(y, sr, segments)\nnotes, durations = merge_consecutive(notes, durations)\ncreate_seq_sheet(notes, durations,"test2_sheet")\nprint(sum(durations), len(y)/sr)\ndf=pd.DataFrame({"notes": notes, "durations": durations})\ndf.to_csv("seq/test2_notes.csv", index=True)\n'

In [102]:
'''
test 3 -> stretching one note
'''
'''files = [
    "keyboard_acoustic_000-050-127.wav",
    "keyboard_acoustic_003-044-127.wav",
    "keyboard_acoustic_007-048-075.wav",
]
def create_test3_file(files):
    combined_audio=[]
    for f in files:
        path = os.path.join("nsynth-train/audio",f)
        y,sr = librosa.load(path, sr=16000)
        if files.index(f)==0:
            y = librosa.effects.time_stretch(y, rate=0.5)  # longer note
        
        '''
        adding silences to make note changes more distinct
        '''
        silence=np.zeros(int(0.2*sr)) #200ms gap

        combined_audio.append(y)
        combined_audio.append(silence)
    return np.concatenate(combined_audio)

y_combined = create_audio_file(files)
sf.write("./seq/test3.wav",y_combined,16000)
y, sr = librosa.load("seq/test3.wav", sr=16000)
segments = segment_audio_onsets(y, sr)
notes, durations = predict_segments(y, sr, segments)
notes, durations = merge_consecutive(notes, durations)
create_seq_sheet(notes, durations,"test3_sheet")
print(sum(durations), len(y)/sr)
df=pd.DataFrame({"notes": notes, "durations": durations})
df.to_csv("seq/test3_notes.csv", index=True)
'''

IndentationError: unexpected indent (2519223621.py, line 18)

In [ ]:
'''
test 4 -> removing silences between notes
'''
'''files = [
    "keyboard_acoustic_000-050-127.wav",
    "keyboard_acoustic_003-044-127.wav",
    "keyboard_acoustic_007-048-075.wav",
]
def create_test4_file(files):
    combined_audio=[]
    for f in files:
        path = os.path.join("nsynth-train/audio",f)
        y,sr = librosa.load(path, sr=16000)
        combined_audio.append(y)

    return np.concatenate(combined_audio)

y_combined = create_test4_file(files)
sf.write("./seq/test4.wav",y_combined,16000)
y, sr = librosa.load("seq/test4.wav", sr=16000)
segments = segment_audio_onsets(y, sr)
notes, durations = predict_segments(y, sr, segments)
notes, durations = merge_consecutive(notes, durations)
create_seq_sheet(notes, durations,"test4_sheet")
print(sum(durations), len(y)/sr)
df=pd.DataFrame({"notes": notes, "durations": durations})
df.to_csv("seq/test4_notes.csv", index=True)
'''

Detected segments: 3 notes 3
Sheet music created at seq/test4_sheet.xml
11.936 12.0


In [107]:
# test 5 -> 6 notes

files = [
    "keyboard_acoustic_000-044-127.wav",
    "keyboard_acoustic_003-048-127.wav",
    "keyboard_acoustic_007-050-075.wav",
    "keyboard_acoustic_011-053-127.wav",
    "keyboard_acoustic_018-045-127.wav",
    "keyboard_acoustic_008-046-075.wav"
]
def create_test5_file(files):
    combined_audio=[]
    for f in files:
        path = os.path.join("nsynth-train/audio",f)
        y,sr = librosa.load(path, sr=16000)
        combined_audio.append(y)

    return np.concatenate(combined_audio)

y_combined = create_test5_file(files)
sf.write("./seq/test5.wav",y_combined,16000)
y, sr = librosa.load("seq/test5.wav", sr=16000)
segments = segment_audio_onsets(y, sr)
notes, durations = predict_segments(y, sr, segments)
notes, durations = merge_consecutive(notes, durations)
create_seq_sheet(notes, durations,"test5_sheet")
print(sum(durations), len(y)/sr)
df=pd.DataFrame({"notes": notes, "durations": durations})
df.to_csv("seq/test5_notes.csv", index=True)

Detected segments: 7 notes 6
Sheet music created at seq/test5_sheet.xml
23.936 24.0


In [112]:
# test 6 -> 8 notes

files = [
    "keyboard_acoustic_000-044-127.wav",
    "keyboard_acoustic_003-048-127.wav",
    "keyboard_acoustic_007-050-075.wav",
    "keyboard_acoustic_011-053-127.wav",
    "keyboard_acoustic_018-045-127.wav",
    "keyboard_acoustic_013-055-075.wav",
    "keyboard_acoustic_016-052-050.wav",
    "keyboard_acoustic_010-056-127.wav"
]

def create_test6_file(files):
    combined_audio=[]
    for f in files:
        path = os.path.join("nsynth-train/audio",f)
        y,sr = librosa.load(path, sr=16000)
        combined_audio.append(y)

    return np.concatenate(combined_audio)

y_combined = create_test6_file(files)
sf.write("./seq/test6.wav",y_combined,16000)
y, sr = librosa.load("seq/test6.wav", sr=16000)
segments = segment_audio_onsets(y, sr)
notes, durations = predict_segments(y, sr, segments)
notes, durations = merge_consecutive(notes, durations)
create_seq_sheet(notes, durations,"test6_sheet")
print(sum(durations), len(y)/sr)
df=pd.DataFrame({"notes": notes, "durations": durations})
df.to_csv("seq/test6_notes.csv", index=True)

Detected segments: 9 notes 8
Sheet music created at seq/test6_sheet.xml
31.936 32.0


In [118]:
# test 7 -> 10 notes
files = [
    "keyboard_acoustic_000-044-127.wav",
    "keyboard_acoustic_003-048-127.wav",
    "keyboard_acoustic_007-050-075.wav",
    "keyboard_acoustic_011-053-127.wav",
    "keyboard_acoustic_018-045-127.wav",
    "keyboard_acoustic_019-047-075.wav",
    "keyboard_acoustic_019-052-100.wav",
    "keyboard_acoustic_010-056-127.wav",
    "keyboard_acoustic_005-049-075.wav",
    "keyboard_acoustic_008-051-127.wav"
]
def create_test7_file(files):
    combined_audio=[]
    for f in files:
        path = os.path.join("nsynth-train/audio",f)
        y,sr = librosa.load(path, sr=16000)
        combined_audio.append(y)

    return np.concatenate(combined_audio)

y_combined = create_test7_file(files)
sf.write("./seq/test7.wav",y_combined,16000)
y, sr = librosa.load("seq/test7.wav", sr=16000)
segments = segment_audio_onsets(y, sr)
notes, durations = predict_segments(y, sr, segments)
notes, durations = merge_consecutive(notes, durations)
create_seq_sheet(notes, durations,"test7_sheet")
print(sum(durations), len(y)/sr)
df=pd.DataFrame({"notes": notes, "durations": durations})
df.to_csv("seq/test7_notes.csv", index=True)

Detected segments: 11 notes 10
Sheet music created at seq/test7_sheet.xml
39.936 40.0
